#### 1. Create the table: **sme_p_business_plan_contract** in Pre-Landed Layer

In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import trim

In [0]:
def load_landed_internal_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

In [0]:
run_date = '2026-02-03'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = '`d-cdh`.landed'
cluster_columns = ["BusinessDate"]
landed_internal_table_path = f"{catalog_landed}.sme_p_business_plan_contract"

# ============================== business_plan_contract_loan ============================

business_plan_contract_loan_extra_filters = None

if business_plan_contract_loan_extra_filters is not None:
    business_plan_contract_loan_extra_filters.append(date_filter)
    business_plan_contract_loan_final_filters = " AND ".join(
        business_plan_contract_loan_extra_filters
        )
else:
    business_plan_contract_loan_final_filters = date_filter

business_plan_contract_loan_selected_fields = [
            "AccountNumber",
            "BusinessPlanContractID",
            "LoanStatus",
        ]

business_plan_contract_loan = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_smebusinessplancontractloan",
            business_plan_contract_loan_final_filters,
            business_plan_contract_loan_selected_fields,
        )

# ============================== business_plan_contract ============================

business_plan_contract_extra_filters = None

if business_plan_contract_extra_filters is not None:
    business_plan_contract_extra_filters.append(date_filter)
    business_plan_contract_final_filters = " AND ".join(
        business_plan_contract_extra_filters
        )
else:
    business_plan_contract_final_filters = date_filter

business_plan_contract_selected_fields = [
            "ID",
            "BusinessPlanStrategyManagementID",
            "AgreementNumber",
            "creationdate",
            "CurrencyCode",
            "BusinessDate"
        ]

business_plan_contract = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_smebusinessplancontract",
            business_plan_contract_final_filters,
            business_plan_contract_selected_fields,
        )

# ============================== business_plan_contract_collateral ============================

business_plan_contract_collateral_extra_filters = None

if business_plan_contract_collateral_extra_filters is not None:
    business_plan_contract_collateral_extra_filters.append(date_filter)
    business_plan_contract_collateral_final_filters = " AND ".join(
        business_plan_contract_collateral_extra_filters
        )
else:
    business_plan_contract_collateral_final_filters = date_filter

business_plan_contract_collateral_selected_fields = [
            "BusinessPlanContractID",
            "ABCollateralID",
            "ThesisCollateralID",
            "OwnershipID",
            "ABPropertyKey",
            "ThesisPropertyKey",
        ]

business_plan_contract_collateral = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_smebusinessplancontractcollateral",
            business_plan_contract_collateral_final_filters,
            business_plan_contract_collateral_selected_fields,
        )


# ============================== joins ============================
sme_business_plan_contract = (
    business_plan_contract.join(
        business_plan_contract_loan,
        business_plan_contract['ID'] == business_plan_contract_loan['BusinessPlanContractID'],
        "left"
        )
        .join(
            business_plan_contract_collateral,
            business_plan_contract_collateral['BusinessPlanContractID'] == business_plan_contract['ID'],
            "left"
            )
)

# ============================== final landed - internal table ============================

sme_business_plan_contract_df = sme_business_plan_contract.select(
    business_plan_contract.BusinessPlanStrategyManagementID.alias("BusinessPlanStrategy_ID"),
    business_plan_contract.AgreementNumber.alias("Agreement_Number"),
    business_plan_contract.creationdate,
    business_plan_contract_loan.AccountNumber.alias("Account_Number"),
    business_plan_contract_collateral.ABCollateralID,
    business_plan_contract_collateral.ThesisCollateralID,
    business_plan_contract_collateral.OwnershipID,
    business_plan_contract_collateral.ABPropertyKey,
    business_plan_contract_collateral.ThesisPropertyKey,
    business_plan_contract.CurrencyCode,
    business_plan_contract_loan.LoanStatus,
    business_plan_contract.BusinessDate
    )